In [1]:
# --- STEP 0: System Setup ---
! git clone https://github.com/aqwertyuiop48/edu-101-java-code.git
print("🔧 Installing Java 17, Temporal CLI, and Maven...")
!sudo apt-get update -y > /dev/null
!sudo apt-get install openjdk-17-jdk-headless maven -y > /dev/null
!ls

fatal: destination path 'edu-101-java-code' already exists and is not an empty directory.
🔧 Installing Java 17, Temporal CLI, and Maven...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
edu-101-java-code  sample_data	samples-java


In [2]:
# 1. Download using the correct 2026 URL
!curl -sSf https://temporal.download/cli.sh | sh

# 2. Add the binary to the Python environment's PATH
import os
temporal_bin_path = os.path.expanduser("~/.temporalio/bin")
os.environ['PATH'] += f":{temporal_bin_path}"

# 3. Create a system-wide link so all '!temporal' commands work seamlessly
!ln -sf /root/.temporalio/bin/temporal /usr/local/bin/temporal

# 4. Verify it works!
print("\n✅ Verification:")
!temporal --version

%cd /content/edu-101-java-code/example1

temporal: Downloading Temporal CLI latest
temporal: Temporal CLI installed at /root/.temporalio/bin/temporal
temporal: For convenience, we recommend adding it to your PATH
temporal: If using bash, run echo export PATH="\$PATH:/root/.temporalio/bin" >> ~/.bashrc

✅ Verification:
temporal version 1.6.1 (Server 1.30.1, UI 2.45.3)
/content/edu-101-java-code/example1


## --- EXAMPLE 1 ---

In [3]:
import subprocess
import time

# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 1...")
# Make sure we are in the right folder!
%cd /content/edu-101-java-code/example1

print("🛰️ Terminal 1: Starting Temporal Server (Port 8081)...")
# Popen starts the server in the background and moves to the next line immediately
import subprocess
import time

# 1. Start the Temporal server in the background (equivalent to trailing '&')
print("Starting Temporal server...")
server_process = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8081','--db-filename','cluster2.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")

# Note: The server_process is still running in the background here.
# You can now run your Gradle command via subprocess, or keep the script running.

print("👷 Terminal 2: Compiling...")
!mvn clean compile > /dev/null

print("👷 Terminal 2: Starting Java Worker...")
# Run the worker in the background too
worker_process = subprocess.Popen(['mvn', 'exec:java', '-Dexec.mainClass=helloworkflow.SayHelloWorker'])

print("⏳ Waiting 15 seconds for server and worker to start...")
time.sleep(15)

print("🎬 Terminal 3: Running Starter...")
# We use ! for the starter because we DO want the notebook to wait for this one to finish
!mvn exec:java -Dexec.mainClass="helloworkflow.Starter"

^C

🏗️ Setting up Example 1...
/content/edu-101-java-code/example1
🛰️ Terminal 1: Starting Temporal Server (Port 8081)...
Starting Temporal server...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
👷 Terminal 2: Compiling...
👷 Terminal 2: Starting Java Worker...
⏳ Waiting 15 seconds for server and worker to start...
🎬 Terminal 3: Running Starter...
[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.example:temporal-demo >----------------------
[INFO] Building temporal-demo 1.0-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- exec-maven-plugin:3.6.3:java (default-cli) @ temporal-demo ---
[helloworkflow.Starter.main()] INFO io.temporal.serviceclient.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}}
Workflow result: Hello Tem

## --- EXAMPLE 2 ---


In [4]:
import subprocess
import time

# Cleanup everything first
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 2...")
# Move to root, clone, then move inside
%cd /content
# Only clone if the folder doesn't exist yet
![ -d "samples-java" ] || git clone https://github.com/temporalio/samples-java
%cd samples-java

print("🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...")
# Popen runs the server in the background without blocking the cell
# Note: I removed '--port 7235' so it defaults to 7233. This allows the sample code to connect!
server_process_2 = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8082','--db-filename','cluster3.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")
print("⏳ Waiting 10 seconds for server to boot...")
time.sleep(10)

print("👷 Terminal 5: Building and Executing HelloAccumulator...")
# We use ! here because we WANT the cell to wait for Gradle to finish and print the result
!./gradlew build > /dev/null
# !./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

^C

🏗️ Setting up Example 2...
/content
/content/samples-java
🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
⏳ Waiting 10 seconds for server to boot...
👷 Terminal 5: Building and Executing HelloAccumulator...
Note: Some input files use or override a deprecated API.
Note: Recompile with -Xlint:deprecation for details.
Note: Some input files use unchecked or unsafe operations.
Note: Recompile with -Xlint:unchecked for details.
/content/samples-java/core/src/test/java/io/temporal/samples/hello/HelloUpdateAndCancellationTest.java:207: warning: [EmptyCatch] Caught exceptions should not be ignored
    } catch (InterruptedException e) {
      ^
    (see https://google.github.io/styleguide/javaguide.html#s6.2-caught-exceptions)
/content/samples-java/core/src/test/java/io/temporal/samples/safemessagepassing/ClusterManagerWorkflowWorkerTest.java:89: warning: [Fu

In [5]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

12:21:47.568 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:21:48.746 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=9328@50e107a6fb2c} 
12:21:48.787 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=9328@50e107a6fb2c} 
Worker started for task queue: HelloAccumulatorTaskQueue
12:21:49.847 {HelloAccumulatorWorkflow-yellow } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - received greeting-Greeting [greetingText=Ted Robot, bucket=yellow, greetingKey=key-4] 
12:21:49.847 {HelloAccumulatorWorkflow-red } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - receiv

In [6]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivity


12:23:57.100 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:23:58.269 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=9968@50e107a6fb2c} 
12:23:58.307 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=9968@50e107a6fb2c} 
12:23:59.313 {HelloActivityWorkflow a42ffaf1-e2cb-32a8-a436-587275511f68} [Activity Executor taskQueue="HelloActivityTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
Hello World!


In [7]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityRetry


12:24:04.645 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:24:05.347 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=10095@50e107a6fb2c} 
12:24:05.376 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=10095@50e107a6fb2c} 
composeGreeting activity is going to fail
12:24:05.964 {HelloActivityWithRetriesWorkflow c7b7e18d-9ca3-3621-b1dd-e4ecba6ab050} [Activity Executor taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=c7b7e18d-9ca3-3621-b1dd-e4ecba6ab050, activityType=ComposeGreeting, attempt=1 

In [8]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityExclusiveChoice


12:24:19.258 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:24:19.868 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=10248@50e107a6fb2c} 
12:24:19.898 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=10248@50e107a6fb2c} 
Order results: Ordered 4 Oranges...Ordered 1 Cherries...Ordered 8 Apples...Ordered 5 Bananas...


In [9]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsync


12:24:24.784 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:24:25.410 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=10366@50e107a6fb2c} 
12:24:25.439 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=10366@50e107a6fb2c} 
Hello World!
Bye World!


In [10]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloParallelActivity


12:24:31.820 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:24:32.832 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=10482@50e107a6fb2c} 
12:24:32.896 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=10482@50e107a6fb2c} 
Hello John!
Hello Mary!
Hello Michael!
Hello Jennet!


In [11]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncActivityCompletion


12:24:38.792 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:24:39.407 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=10611@50e107a6fb2c} 
12:24:39.427 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=10611@50e107a6fb2c} 
Hello World!


In [12]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncLambda


12:24:43.948 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:24:44.577 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=10732@50e107a6fb2c} 
12:24:44.596 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=10732@50e107a6fb2c} 
Hello World!
Hello World!


In [13]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDetachedCancellationScope


12:24:51.453 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:24:52.495 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=10856@50e107a6fb2c} 
12:24:52.541 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=10856@50e107a6fb2c} 
12:24:55.060 {HelloDetachedCancellationWorkflow 2e1e7b08-97aa-363d-b7fb-b7d1362467bb} [Activity Executor taskQueue="HelloDetachedCancellationTaskQueue", namespace="default": 1] INFO  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity canceled. ActivityId=2e1e7b08-97aa-363d-b7fb-b7d1362467bb, activityType=SayHello, attempt=1 
Goodbye John!


In [14]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloChild


12:24:59.027 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:24:59.673 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloChildTaskQueue", namespace="default", identity=10988@50e107a6fb2c} 
Hello World!


In [15]:
!timeout 240s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloCron
# taking too long

12:25:06.292 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:25:07.477 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=11098@50e107a6fb2c} 
12:25:07.524 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=11098@50e107a6fb2c} 
Started workflow_id: "HelloCronWorkflow"
run_id: "019cd7b5-478f-7f14-b947-366c756aa4ec"



From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!


In [16]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDynamic


12:29:04.946 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:29:05.646 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=12185@50e107a6fb2c} 
12:29:05.690 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=12185@50e107a6fb2c} 
DynamicACT: Hello John from: DynamicWF


In [17]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloEagerWorkflowStart


12:29:12.690 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:29:13.365 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=12307@50e107a6fb2c} 
12:29:13.384 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=12307@50e107a6fb2c} 
12:29:13.981 {HelloEagerWorkflowStartWorkflow 38be69aa-4648-33f4-9986-c7bc42ef1c29} [Activity Executor taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloEagerWorkflowStart$GreetingLocalActivitiesImpl - Composing greeting... 
Hello World!


In [18]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloException


12:29:18.385 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:29:19.073 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=12426@50e107a6fb2c} 
12:29:19.100 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=12426@50e107a6fb2c} 
12:29:19.810 {45b52911-457a-332e-84cd-f4d06c0965dc 8463518b-5da4-3c49-a492-94d93565595b} [Activity Executor taskQueue="HelloExceptionTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=8463518b-5da4-3c49-a492-94d93565595b, activityType=ComposeGreeting, attempt=1 
java.io.IOException: Hello World!
	at io.temporal.samples.hello.Hel

In [19]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloLocalActivity


12:29:27.771 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:29:28.750 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloLocalActivity", namespace="default", identity=12558@50e107a6fb2c} 
12:29:28.769 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloLocalActivity", namespace="default", identity=12558@50e107a6fb2c} 
Hello World!


In [20]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPeriodic


12:29:33.528 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:29:34.132 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=12681@50e107a6fb2c} 
12:29:34.171 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=12681@50e107a6fb2c} 
Started a new GreetingWorkflow.
Observing the workflow execution for 20 seconds.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5410 milliseconds and then I will greet you again.


From HelloPeriodicWorkflow: Hello World! I will sleep for 4145 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 4525 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 4831 milliseconds and then I will greet you again.
Requesting the workflow to exit.
From HelloPeriodicWorkflow: Hello World! I will sleep for 4063 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! It was requested to quit the periodic greetings, so this the last one.
Good bye.


In [21]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPolymorphicActivity


12:29:59.917 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:30:00.584 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=12882@50e107a6fb2c} 
12:30:00.619 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=12882@50e107a6fb2c} 
Hello World!
Bye World!



In [22]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloQuery


12:30:05.056 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:30:05.679 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloQueryTaskQueue", namespace="default", identity=13000@50e107a6fb2c} 
Hello World!
Bye World!


In [23]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSaga


12:30:14.842 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:30:15.510 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=13123@50e107a6fb2c} 
12:30:15.536 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=13123@50e107a6fb2c} 
ActivityOperationImpl.execute() is called with amount 10
ActivityOperationImpl.execute() is called with amount 20
Other compensation logic in main workflow.
ActivityCompensationImpl.compensate() is called with amount -20
ActivityCompensationImpl.compensate() is called with amount -10


In [24]:
!timeout 300s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSchedules

12:30:21.257 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:30:21.890 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=13252@50e107a6fb2c} 
12:30:21.916 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=13252@50e107a6fb2c} 
From HelloScheduleWorkflow-2026-03-10T12:30:22Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:30:22Z!
From HelloScheduleWorkflow-2026-03-10T12:30:25Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:30:25Z!
From HelloScheduleWorkflow-2026-03-10T12:30:30Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:30:30Z!
From HelloScheduleWorkflow-2026-03-10T12:30:35Z: Hello World

From HelloScheduleWorkflow-2026-03-10T12:30:40Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:30:40Z!
From HelloScheduleWorkflow-2026-03-10T12:30:45Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:30:45Z!
From HelloScheduleWorkflow-2026-03-10T12:30:50Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:30:50Z!
From HelloScheduleWorkflow-2026-03-10T12:30:55Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:30:55Z!
From HelloScheduleWorkflow-2026-03-10T12:31:00Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:31:00Z!
From HelloScheduleWorkflow-2026-03-10T12:31:05Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:31:05Z!
From HelloScheduleWorkflow-2026-03-10T12:31:10Z: Hello World from HelloSchedule scheduled at 2026-03-10T12:31:10Z!


In [25]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignal


12:31:16.468 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:31:17.116 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalTaskQueue", namespace="default", identity=13569@50e107a6fb2c} 
[Hello World!, Hello Universe!]


In [26]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSearchAttributes

12:31:21.734 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:31:22.950 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=13680@50e107a6fb2c} 
12:31:22.990 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=13680@50e107a6fb2c} 
In workflow we get CustomKeywordField is: keys
Hello SearchAttributes!


In [27]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloTypedSearchAttributes


12:31:29.425 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:31:30.058 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=13807@50e107a6fb2c} 
12:31:30.094 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=13807@50e107a6fb2c} 
Hello TypedSearchAttributes how are you doing?


In [28]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSideEffect



12:31:35.483 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 


12:31:36.180 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=13922@50e107a6fb2c} 
12:31:36.220 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=13922@50e107a6fb2c} 
Hello World
Hello World


In [29]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloUpdate


12:31:42.760 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:31:43.905 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=14044@50e107a6fb2c} 
12:31:43.945 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=14044@50e107a6fb2c} 
12:31:45.080 {HelloUpdateWorkflow af13684e-e373-3475-97d7-fc28aafb2846} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
12:31:45.240 {HelloUpdateWorkflow 501c1636-7188-319f-b1e1-e11d46ac819e} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$Greetin

In [30]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithTimer


12:31:50.276 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:31:50.936 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=14174@50e107a6fb2c} 
12:31:50.954 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=14174@50e107a6fb2c} 
Processing value: Value 2
Processing value: Value 5
Processing value: Value 7
Processing value: Value 10
12:32:16.492 {HelloSignalWithTimerWorkflow } [workflow-method-HelloSignalWithTimerWorkflow-8e0d85ea-513a-4896-bdcc-431be7ad5f60] INFO  i.t.s.h.HelloSignalWithTimer$SignalWithTimerWorkflowImpl - Timer canceled via exit signal 
Processing value: Value 11


In [31]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithStartAndWorkflowInit

12:32:20.290 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
12:32:20.886 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=14391@50e107a6fb2c} 
12:32:20.915 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=14391@50e107a6fb2c} 
Result: Hello Michael Jordan,Hello John Stockton
12:32:21.889 {without-init } [signal addGreeting] WARN  i.t.i.sync.WorkflowExecutionHandler - Workflow execution failure WorkflowId='without-init', RunId=ad2bc145-f920-473d-94c8-b1f450859e88, WorkflowType='MyWorkflowNoInit' 
java.lang.NullPointerException: Cannot invoke "java.util.List.add(Object)" because "this.peopleToGreet" is null
	at io.temporal.sam

In [32]:
# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

^C
